# State of Data Brasil — Pipeline AWS
Pipeline completo: Bronze → Silver → Gold com PySpark no AWS Glue

In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
print('✅ Sessão Spark iniciada!')
print(f'Versão Spark: {spark.version}')

In [ ]:
# CAMADA BRONZE — Leitura dos dados brutos
bucket = 'state-of-data-brasil'

df_2021 = spark.read.option('header','true').option('inferSchema','true').csv(f's3://{bucket}/bronze/State of Data 2021 - Dataset - Pgina1.csv')
df_2022 = spark.read.option('header','true').option('inferSchema','true').csv(f's3://{bucket}/bronze/State_of_data_2022.csv')
df_2023 = spark.read.option('header','true').option('inferSchema','true').csv(f's3://{bucket}/bronze/State_of_data_BR_2023_Kaggle - df_survey_2023.csv')

print('✅ Dados carregados!')
print(f'2021: {df_2021.count()} linhas')
print(f'2022: {df_2022.count()} linhas')
print(f'2023: {df_2023.count()} linhas')

In [ ]:
# CAMADA SILVER — Limpeza e padronização
from pyspark.sql import functions as F

def find_col(df, search):
    for c in df.columns:
        if search in c:
            return c
    return None

def select_cols(df, ano):
    def fc(s): return find_col(df, s)
    raca = fc("'Cor/raca/etnia'")
    cols = [
        F.col("`"+fc("'Idade'")+"`").alias('idade'),
        F.col("`"+fc("'Genero'")+"`").alias('genero'),
        F.col("`"+fc("Nivel de Ensino")+"`").alias('escolaridade'),
        F.col("`"+fc("'Cargo Atual'")+"`").alias('cargo'),
        F.col("`"+fc("'Faixa salarial'")+"`").alias('faixa_salarial'),
        F.col("`"+fc("'Estado onde mora'")+"`").alias('estado'),
        F.col("`"+fc("', 'Nivel')")+"`").alias('senioridade'),
        F.lit(ano).alias('ano_pesquisa')
    ]
    if raca:
        cols.insert(5, F.col("`"+raca+"`").alias('raca_etnia'))
    else:
        cols.insert(5, F.lit(None).cast('string').alias('raca_etnia'))
    return df.select(cols)

df_silver = select_cols(df_2021,2021).unionByName(select_cols(df_2022,2022)).unionByName(select_cols(df_2023,2023))
df_silver = df_silver.dropna(subset=['cargo','faixa_salarial','genero'])
df_silver.write.mode('overwrite').parquet(f's3://{bucket}/silver/state_of_data_silver/')
print(f'✅ Silver criada! Total: {df_silver.count():,} registros')

In [ ]:
# CAMADA GOLD — Análises agregadas
df_silver.groupBy('ano_pesquisa').count().orderBy('ano_pesquisa').write.mode('overwrite').parquet(f's3://{bucket}/gold/crescimento/')
df_silver.groupBy('ano_pesquisa','cargo').count().write.mode('overwrite').parquet(f's3://{bucket}/gold/cargos/')
df_silver.groupBy('ano_pesquisa','genero').count().write.mode('overwrite').parquet(f's3://{bucket}/gold/genero/')
df_silver.groupBy('ano_pesquisa','faixa_salarial').count().write.mode('overwrite').parquet(f's3://{bucket}/gold/salario/')
df_silver.groupBy('ano_pesquisa','senioridade').count().write.mode('overwrite').parquet(f's3://{bucket}/gold/senioridade/')
print('✅ Camada Gold salva!')

In [ ]:
# VISUALIZAÇÕES
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('State of Data Brasil 2021-2023', fontsize=16, fontweight='bold')

anos = [2021, 2022, 2023]
respondentes = [1857, 2975, 3857]
axes[0,0].bar(anos, respondentes, color=['#185FA5','#1D9E75','#E85D24'], width=0.5)
axes[0,0].set_title('Crescimento do Mercado')

axes[0,1].pie([2896,943,18], labels=['Masculino','Feminino','Outros'],
              colors=['#185FA5','#E85D24','#1D9E75'], autopct='%1.1f%%')
axes[0,1].set_title('Diversidade de Gênero (2023)')

axes[1,0].barh(['Analista BI','Eng. Dados','Cientista','Analista Dados'],
               [506,684,687,907], color='#185FA5')
axes[1,0].set_title('Top Cargos (2023)')

axes[1,1].bar(['Júnior','Pleno','Sênior'], [1046,1392,1418],
              color=['#EF9F27','#1D9E75','#185FA5'], width=0.5)
axes[1,1].set_title('Senioridade (2023)')

plt.tight_layout()
%matplot plt